In [1]:
import psi4
import pandas as pd
import os
import numpy as np
from lps_rscf import lps_solver

In [2]:
csv_file = 'g_test_closed_shell_atoms.csv'

if os.path.exists(csv_file):
    df = pd.read_csv(csv_file)
    print(f"-> Loaded existing results: {len(df)} rows found.")
else:
    df = pd.DataFrame()
    print("-> No existing file found. Starting fresh.")

-> No existing file found. Starting fresh.


In [ ]:
psi4.core.set_output_file('output.dat', False)

ATOMS = {
    'He':  {'mult': 1,'N': 2}, 
    # 'Be': {'mult': 1,'N': 4},
    # 'Ne': {'mult': 1,'N': 10}, 
    # 'Mg': {'mult': 1,'N': 12},
    # 'Ar':  {'mult': 1,'N': 18}, 
    # 'Ca':  {'mult': 1,'N': 20},
    # 'Zn':  {'mult': 1,'N': 30}, 
    # 'Kr':  {'mult': 1,'N': 36}
}

METHOD = "TFW FA"
TP = ['LDA_K_TF', 1.0]
LAMBDA = 1.0
EXC = ['LDA_X', 0.0, 'LDA_C_VWN', 0.0]
FA = [True, 1.0]
DIIS = True
MAX_ITER = 100
DAMPING = [0.9, 0.0, 0.001]
D_guess = None
verbose=True

psi4.set_options({'basis': 'UGBS_S', 
                  'DFT_SPHERICAL_POINTS': 6, 
                  'DFT_RADIAL_POINTS': 1000})

for atom in ATOMS:
    
    if not df.empty:
        exists = df[
            (df['Atom'] == atom) & 
            (df['Method'] == METHOD) & 
            (df['Basis'] == psi4.core.get_global_option("BASIS"))
        ]
        if not exists.empty:
            print(f"Skipping {atom} (Already exists for {METHOD}/{psi4.core.get_global_option("BASIS")})")
            continue

    print(f"Calculating {atom} with {METHOD}...")
    MOL = psi4.geometry(f"0 {ATOMS[atom]['mult']}\n {atom}\nsymmetry c1")
    try:
        E, D, mu, iterations = lps_solver(MAX_ITER,TP,EXC,LAMBDA,MOL,DAMPING,FA,D_guess,DIIS,verbose)
        if iterations >= MAX_ITER:
            print("  !!! SCF failed to converge (Max cycles exceeded).")
        else:
            print(f"Calculated Energy: {E:.4f} Hartree")
            # row = {
            #     "Method": METHOD,
            #     "Atom": atom,
            #     "Basis": psi4.core.get_global_option("BASIS"),
            #     "Grid_Sph": psi4.core.get_global_option("DFT_SPHERICAL_POINTS"),
            #     "Grid_Rad": psi4.core.get_global_option("DFT_RADIAL_POINTS"),
            #     "Energy,Ha": round(E, 6),
            #     "ChemPot,Ha": round(mu, 6),
            #     "Iterations": iterations,
            #     "DIIS": DIIS,
            #     "Damp_Start": DAMPING[0],
            #     "Damp_End": DAMPING[1],
            #     "Damp_Cutoff": DAMPING[2]
            # }
            
            # df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    
    except Exception as e:
        print(f"  !!! Failed {atom}. Error: {e}")
        continue